# AG News - classification multiclasse

120 000 titres de presse en 4 categories (World, Sports, Business, Sci/Tech). Difference avec IMDB : sortie softmax a 4 unites + loss sparse_categorical_crossentropy.
Le Tokenizer de Keras 2 a ete retire en Keras 3 : on utilise la couche TextVectorization a la place.

In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import numpy as np
import keras
from keras import layers
from keras.layers import TextVectorization
from datasets import load_dataset
from sklearn.metrics import classification_report
import tensorflow as tf
tf.get_logger().setLevel('ERROR')
keras.utils.set_random_seed(42)

VOCAB_SIZE_AG = 20000
MAX_LEN_AG = 50
EMBED_DIM_AG = 64
LABELS = ['World', 'Sports', 'Business', 'Sci/Tech']

In [2]:
dataset = load_dataset('fancyzhx/ag_news')
train_texts = dataset['train']['text']
train_labels = np.array(dataset['train']['label'])
test_texts = dataset['test']['text']
test_labels = np.array(dataset['test']['label'])
print('train:', len(train_texts), '| test:', len(test_texts))
print('distribution train:', np.bincount(train_labels))
print('exemple:', train_texts[0][:80], '-> label', train_labels[0], f'({LABELS[train_labels[0]]})')

train: 120000 | test: 7600
distribution train: [30000 30000 30000 30000]
exemple: Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall  -> label 2 (Business)


In [3]:
# TextVectorization : construit le vocab et transforme le texte en entiers (remplace le Tokenizer)
vectorizer = TextVectorization(max_tokens=VOCAB_SIZE_AG, output_sequence_length=MAX_LEN_AG, output_mode='int')
vectorizer.adapt(train_texts)  # vocab construit sur le train uniquement (pas de fuite du test)
X_train_ag = vectorizer(train_texts).numpy()
X_test_ag = vectorizer(test_texts).numpy()
print('X_train_ag:', X_train_ag.shape, '| X_test_ag:', X_test_ag.shape)

X_train_ag: (120000, 50) | X_test_ag: (7600, 50)


In [4]:
model_ag = keras.Sequential([
    layers.Input(shape=(MAX_LEN_AG,)),
    layers.Embedding(input_dim=VOCAB_SIZE_AG, output_dim=EMBED_DIM_AG),
    layers.LSTM(64),
    layers.Dropout(0.3),
    layers.Dense(4, activation='softmax'),
])
model_ag.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model_ag.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 50, 64)         │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 4)              │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,313,284 (5.01 MB)

 Trainable params: 1,313,284 (5.01 MB)

 Non-trainable params: 0 (0.00 B)

In [5]:
history_ag = model_ag.fit(
    X_train_ag, train_labels,
    epochs=3,
    batch_size=256,
    validation_split=0.05,
    verbose=2,
)

Epoch 1/3


446/446 - 42s - 93ms/step - accuracy: 0.8054 - loss: 0.4987 - val_accuracy: 0.8982 - val_loss: 0.3005


Epoch 2/3


446/446 - 39s - 87ms/step - accuracy: 0.9213 - loss: 0.2520 - val_accuracy: 0.8997 - val_loss: 0.2982


Epoch 3/3


446/446 - 40s - 90ms/step - accuracy: 0.9360 - loss: 0.2075 - val_accuracy: 0.9050 - val_loss: 0.2930


In [6]:
loss_ag, acc_ag = model_ag.evaluate(X_test_ag, test_labels, verbose=0)
print(f'Accuracy AG News test : {acc_ag:.4f}')
y_pred_ag = np.argmax(model_ag.predict(X_test_ag, verbose=0), axis=1)
print()
print(classification_report(test_labels, y_pred_ag, target_names=LABELS))

Accuracy AG News test : 0.9128



              precision    recall  f1-score   support

       World       0.93      0.90      0.92      1900
      Sports       0.96      0.97      0.97      1900
    Business       0.87      0.89      0.88      1900
    Sci/Tech       0.89      0.89      0.89      1900

    accuracy                           0.91      7600
   macro avg       0.91      0.91      0.91      7600
weighted avg       0.91      0.91      0.91      7600



In [7]:
def predict_ag_news(text):
    probs = model_ag.predict(vectorizer([text]), verbose=0)[0]
    return LABELS[int(np.argmax(probs))], {LABELS[i]: round(float(probs[i]), 3) for i in range(4)}

for title in ['Tesla announces record quarterly profits',
              'Olympics 2024: Tech companies sponsor all major sports events',
              'Scientists discover a new exoplanet near a distant star']:
    label, probs = predict_ag_news(title)
    print(f'{label:10s} | {title}')
    print(f'           {probs}')

Business   | Tesla announces record quarterly profits
           {'World': 0.024, 'Sports': 0.001, 'Business': 0.743, 'Sci/Tech': 0.231}
Sports     | Olympics 2024: Tech companies sponsor all major sports events
           {'World': 0.136, 'Sports': 0.826, 'Business': 0.016, 'Sci/Tech': 0.022}
Sci/Tech   | Scientists discover a new exoplanet near a distant star
           {'World': 0.162, 'Sports': 0.009, 'Business': 0.105, 'Sci/Tech': 0.723}
